# 02 PyPSA

[PyPSA](https://pypsa.org) builds its optimisation problem with linopy, so everything from
[01a linopy](01a-linopy.ipynb) applies: you keep working in PyPSA, and the block structure comes
from splitting the `snapshot` dimension. Solving is `n.optimize()` with two more arguments.

| sections | run on |
|---|---|
| 1 to 5: build the network, see where its variables go, export, visualize, change the network, reference solve with HiGHS | every machine |
| 6: solve with PIPS-IPM++ | Linux or WSL with the `solver` extra |
| 7: more than one process with MPI | Linux or WSL with the `solver` extra |

In [ ]:
import workshop

workshop.check()

<details><summary>💡 Hint: why PyPSA 1.2 and not the latest?</summary>

The PIPS-IPM++ backend lives on a fork of linopy 0.8, and PyPSA 1.3 requires linopy 0.9. The
workshop therefore pins `pypsa>=1.2.2,<1.3`. If you install into another environment yourself,
keep that pin: without it the installer quietly picks a PyPSA from 2022 instead of reporting the
conflict.
</details>

## 1. A small two-region network

Two buses, wind and gas at each, a link between them, and a CO2 limit: the sample model from
[00](00-start-here.ipynb) in PyPSA's own vocabulary. The numbers come from `sample_data.py`, the
same file the 01 notebooks use. The south has a little more demand, and its profiles are shifted
by three hours, so the link has something to do.

The cell below writes the network into a file, `pypsa_model.py`, so this notebook and the parallel
run in section 7 use the same code. The cell after it loads the file.

In [ ]:
%%writefile pypsa_model.py
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa

from sample_data import BUDGET_PER_DAY, SOLAR, STORAGE, TECHS, profiles

BUSES = ["north", "south"]

# the PIPS-IPM++ backend of the linopy fork aligns all variables once to find the block
# dimension, which warns for PyPSA models whose components have different names; harmless
warnings.filterwarnings("ignore", "Coordinates across variables not equal")
logging.getLogger("pypsa").setLevel(logging.WARNING)
logging.getLogger("linopy").setLevel(logging.WARNING)


def build(T=24, solar=False, storage=False, cyclic=False):
    n = pypsa.Network()
    n.set_snapshots(pd.RangeIndex(T))
    techs = TECHS | ({"solar": SOLAR} if solar else {})
    n.add("Carrier", "AC")
    for name, (_, _, emission, _) in techs.items():
        n.add("Carrier", name, co2_emissions=emission)

    demand, avail = profiles(T)
    for i, bus in enumerate(BUSES):
        n.add("Bus", bus, carrier="AC")
        n.add("Load", f"load_{bus}", bus=bus, p_set=demand * (1.0 + 0.3 * i))
        for name, (investment, variable, _, profile) in techs.items():
            n.add("Generator", f"{name}_{bus}", bus=bus, carrier=name, p_nom_extendable=True,
                  capital_cost=investment, marginal_cost=variable,
                  p_max_pu=np.roll(avail[profile], 3 * i))
        if storage:
            n.add("StorageUnit", f"battery_{bus}", bus=bus, p_nom_extendable=True,
                  capital_cost=STORAGE["investment"] * 4, max_hours=4,
                  efficiency_store=STORAGE["efficiency"], cyclic_state_of_charge=cyclic)
    n.add("Link", "north_south", bus0="north", bus1="south", carrier="AC",
          p_nom_extendable=True, capital_cost=0.1, p_min_pu=-1)
    # the emission budget: a linking row, as in every other notebook
    n.add("GlobalConstraint", "co2", carrier_attribute="co2_emissions", sense="<=",
          constant=len(BUSES) * BUDGET_PER_DAY * (T / 24))
    return n

In [ ]:
%run pypsa_model.py

n = build()
n

<details><summary>💡 Hint: I changed the network. Why did nothing happen?</summary>

`%%writefile` saves the code to `pypsa_model.py`, and `%run` loads it. After editing the first,
run both again, and then `build()` gives you the new network.
</details>

## 2. Where do the blocks come from?

`n.optimize.create_model()` builds the linopy model without solving it. Its variables tell you
where they go: a variable with a `snapshot` dimension is split across the leaves, one without it
stays in the root.

In [ ]:
n = build()
n.optimize.create_model(include_objective_constant=False)

print(f"{'variable':22s} {'per snapshot':>13s} {'size':>5s}   goes to")
for name, var in n.model.variables.items():
    timed = "snapshot" in var.dims
    print(f"{name:22s} {str(timed):>13s} {var.size:>5d}   {'a leaf' if timed else 'the root'}")

This is why PyPSA suits PIPS-IPM++ so well: the investment variables (`p_nom`) are few and shared,
the dispatch variables are many and local to their snapshot. As you add snapshots the root stays
the same size, and snapshots are exactly the direction energy models grow in.

<details><summary>💡 Hint: what is <code>include_objective_constant=False</code> for?</summary>

PyPSA 1.2 warns that the default of this argument will change in 2.0. It controls whether a
constant term (costs of fixed assets) is carried in the objective. This network has none, so the
value is the same either way. Setting it explicitly silences the warning, and `False` is what
PyPSA recommends for numerical conditioning.
</details>

## 3. Export it and look at the matrix

`n.model` is an ordinary linopy model, so the linopy writer from 01a exports it. It needs neither
the solver nor MPI.

In [ ]:
from pathlib import Path
from linopy.solvers import PIPSIPMpp

Path("data").mkdir(exist_ok=True)


def export(n, name, n_blocks=4):
    n.optimize.create_model(include_objective_constant=False)
    return PIPSIPMpp.write_parquet(n.model, f"data/{name}", layout="monolithic",
                                   n_blocks=n_blocks, names=True)


workshop.visualize(export(build(), "pypsa_base"))

How to read it:

- The narrow band on the left holds the 5 root columns: four generator capacities and the link
  capacity.
- The four coloured squares are the leaf blocks, six snapshots each. Every snapshot has four
  generator dispatch variables and one link flow, and the rows of the nodal balances and the
  dispatch limits.
- The grey row at the bottom is the one linking row, the CO2 limit `GlobalConstraint-co2`.

## 4. Change the network and watch the matrix

Each exercise changes the network once. Before you run a cell, predict what happens to the three
numbers: root columns, columns per leaf, linking rows.

### Exercise 1: more snapshots

Two days in 8 blocks:

In [ ]:
workshop.visualize(export(build(T=48), "pypsa_T48_8b", n_blocks=8))

Twice the snapshots, twice the leaves, the same leaf size and the same 5 root columns.

### Exercise 2: another technology

`build(solar=True)` adds a solar generator at each bus, with the `SOLAR` row from
`sample_data.py`.

In [ ]:
workshop.visualize(export(build(solar=True), "pypsa_solar"))

Two more root columns, one per new generator, and every snapshot gains two dispatch variables.

To add a technology of your own, extend `techs` in the model file, for example
`TECHS | {"solar": SOLAR, "biogas": (0.6, 0.3, 0.1, "firm")}`, run both cells of section 1 again,
and export. (Editing `sample_data.py` works too, but then restart the kernel: Python keeps the
module it imported first.)

### Exercise 3: storage

`build(storage=True)` adds a `StorageUnit` at each bus. Its state of charge in snapshot `t`
depends on snapshot `t - 1`. Predict the linking rows before you run:

In [ ]:
workshop.visualize(export(build(storage=True), "pypsa_storage"))

Seven linking rows: the CO2 limit, plus one state-of-charge row per storage unit and block
boundary, 2 x 3 = 6. All six are 2-link: they touch only two neighbouring blocks, which
PIPS-IPM++ handles cheaply. The two storage capacities join the root.

### Exercise 4: cyclic storage

PyPSA storage units have a switch that ties the last snapshot back to the first:
`cyclic_state_of_charge=True`. What does it do to the borders?

In [ ]:
workshop.visualize(export(build(storage=True, cyclic=True), "pypsa_cyclic"))

Two more linking rows, one per storage unit, and they are not 2-link: they tie block 4 back to
block 1, which are not neighbours. With two storage units that is harmless. In a large network
with thousands of cyclic storage units it becomes a real cost, and it is worth asking whether the
model needs the cycle over the whole horizon.

### Exercise 5: your own change

- Split into 24 blocks, one per snapshot. How many of the storage rows link now?
- Add a second link, or a third bus. Where do the new variables go?
- A split other than contiguous slices: set `n.model.blocks` after `create_model`, as in
  exercise 4 of [01a](01a-linopy.ipynb), and export without `n_blocks`.

## 5. The answer, with HiGHS

A reference answer that runs everywhere:

In [ ]:
reference = {}
for label, kwargs in (("base", {}), ("storage", {"storage": True})):
    net = build(**kwargs)
    net.optimize(solver_name="highs", solver_options={"output_flag": False},
                 include_objective_constant=False)
    reference[label] = net.objective
    print(f"{label:8s} objective {net.objective:.6f}")
    print(net.generators.p_nom_opt.round(4).to_string(), "\n")

## 6. Solve with PIPS-IPM++

From here on the cells need the solver: Linux or WSL, with JupyterLab started through
`uv run --extra solver jupyter lab` or `pixi run -e solver lab`. On other machines they are
skipped.

Two arguments carry the block structure; `solver_options` go to PIPS-IPM++ itself.

In [ ]:
%%requires solver
solved = {}
for label, kwargs in (("base", {}), ("storage", {"storage": True})):
    net = build(**kwargs)
    status, condition = net.optimize(
        solver_name="pipsipmpp",
        solver_options={"SILENT": True},
        n_blocks=4,
        block_dim="snapshot",
        include_objective_constant=False,
    )
    solved[label] = (status, condition, net.objective)

In [ ]:
%%requires solver
for label, (status, condition, objective) in solved.items():
    difference = abs(objective - reference[label]) / abs(reference[label])
    print(f"{label:8s} {status} {condition}  objective {objective:.6f}  "
          f"(HiGHS {reference[label]:.6f}, relative difference {difference:.1e})")

The results land on the network as after any other solve (`p_nom_opt`, `n.generators_t.p`,
`n.buses_t.marginal_price`), so everything you already do with a solved network keeps working.

The two solvers agree to about five or six figures, not to the last digit, and that is expected.
PIPS-IPM++ is an interior-point method and stops once it is within a tolerance of the optimum;
HiGHS ends on an exact simplex vertex by default. For capacity expansion, differences at that
scale are far below the uncertainty in any input data.

<details><summary>💡 Hint: can I make them agree more closely?</summary>

Tighten the convergence tolerance of PIPS-IPM++ through `solver_options` (see the option list of
the solver for the tolerance names). It costs iterations. Checking against a reference solver
once, on a model you care about, is good practice; tightening further rarely is.
</details>

## 7. More than one process

In [ ]:
%%writefile pypsa_parallel.py
from mpi4py import MPI
from pypsa_model import build

n = build(storage=True)
status, condition = n.optimize(solver_name="pipsipmpp", solver_options={"SILENT": True},
                               n_blocks=4, block_dim="snapshot",
                               include_objective_constant=False)
if MPI.COMM_WORLD.Get_rank() == 0:
    print(f"ranks={MPI.COMM_WORLD.Get_size()} {status} {condition} objective={n.objective:.6f}")

In [ ]:
%%requires solver
for ranks in (1, 2, 4):
    workshop.mpirun("pypsa_parallel.py", ranks, show="ranks=")

The same objective on one, two and four processes, within the solver tolerance. The work is split
differently, so the last digits can differ.

Every process builds the network, which is cheap, and the backend uses the model from process 0
only. What you must not do is call `optimize()` on only some processes: it is collective.

## 8. Now your own network

```python
n = pypsa.Network("my-network.nc")
n.optimize.create_model(include_objective_constant=False)
PIPSIPMpp.write_parquet(n.model, "data/mine", layout="monolithic", n_blocks=12, names=True)
workshop.visualize("data/mine")
```

and on Linux

```python
n.optimize(solver_name="pipsipmpp", n_blocks=12, block_dim="snapshot")
```

Questions worth asking:

1. How many snapshots? Blocks pay off when each leaf is substantial. A year at hourly resolution
   in 12 monthly blocks is a sensible start; 8760 one-hour blocks is not.
2. What is in your root? Every extendable `p_nom`, plus any variable without a snapshot
   dimension. Global constraints such as a CO2 cap land in the bottom border. Many investment
   variables, from a high spatial resolution, make the root large, and that is the limit of this
   decomposition.
3. Cyclic storage? Each cyclic unit adds a row that links the last block to the first.
4. Multi-period investment? Splitting on `period` instead of `snapshot` is often the better
   axis: `block_dim="period"`.

A large network visualizes slowly. `workshop.structure("data/mine")` gives the numbers without
the picture.

## 9. Checkpoint

- Which PyPSA variables end up in the root, and why is that good news as you add snapshots?
- Why does a `StorageUnit` create linking rows, and why only a few?
- What does `cyclic_state_of_charge=True` add, and why is it not a 2-link row?

## Next

[03 Structure detection](03-structure-detection.ipynb): finding a structure in a model file that
states none, on a real energy system model.

## Notes

- LP only. Anything with binaries, such as `committable=True` unit commitment, is out of scope;
  PIPS-IPM++ does not branch.
- `n.model` is the underlying linopy model, so everything in [01a](01a-linopy.ipynb) applies,
  including `Model.blocks` for splits other than contiguous slices.